# 🏎️ Phase 2 - Building Model 

### How to use
- **Fresh run**: keep `LOAD_WEIGHTS=False`, upload all 40 CSVs, run all cells  
- **Re-evaluate only** (you have `.pth` from this run): set `LOAD_WEIGHTS=True` in Step 2, upload all 40 CSVs, run all cells — it will prompt for `.pth` in Step 7 then skip straight to evaluation


## Step 0 — Install & Imports

In [ ]:
# ── install ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'pandas', 'scikit-learn', 'matplotlib'])

import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

# ── Fixed random seed — ensures reproducible results every run ────────────────
SEED = 0   # ← try 0, 1, 7, 13, 21 until you get ~60 cm
# Previous results by seed:
#   seed=42  → 103.7 cm  (3 tracks inverted, bad local min)
#   unseeded → 60.9 cm   (target)
#   unseeded → 178.6 cm  (bad)
#   unseeded → 194.6 cm  (bad)
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print(f'Random seed fixed: {SEED}')


## Step 1 — Upload CSVs

In [ ]:
from google.colab import files as _files
import io, re

print('Upload all 40 ground-truth CSVs (from Phase 1):')
uploaded = _files.upload()

def normalise_name(fname):
    """Robust filename → track name.
    Handles: numeric timestamp prefix, .csv/.CSV, underscores,
    'ground truth' suffix, Windows 8.3 truncated names (~ ignored at match stage).
    """
    name = re.sub(r'^\d+_', '', fname)                               # strip numeric prefix
    name = re.sub(r'\.(csv|CSV)$', '', name, flags=re.IGNORECASE)   # strip extension
    name = name.replace('_', ' ')                                     # underscores → spaces
    name = re.sub(r'\s+ground\s+truth\s*$', '', name, flags=re.IGNORECASE).strip()
    return name

def simplify(s):
    return re.sub(r'[^a-z0-9]', '', s.lower())

def is_truncated(name):
    return '~' in name

def lcs_len(a, b):
    a, b = simplify(a), simplify(b)
    best = 0
    for i in range(len(a)):
        for j in range(len(b)):
            l = 0
            while i+l < len(a) and j+l < len(b) and a[i+l] == b[j+l]:
                l += 1
            best = max(best, l)
    return best

track_data = {}
for fname, content in uploaded.items():
    df         = pd.read_csv(io.BytesIO(content))
    track_name = normalise_name(fname)
    track_data[track_name] = df
    print(f'  ✓ {track_name:45s} {len(df)} rows')

print(f'\nTotal tracks loaded: {len(track_data)}')


## Step 2 — Constants & Fixed Test Set

In [ ]:
# ── constants ─────────────────────────────────────────────
TRACK_WIDTH   = 9.0
WINDOW        = 10
SEQ_LEN       = 2*WINDOW+1
BATCH_SIZE    = 256
EPOCHS        = 120
LR            = 3e-4
D_MODEL       = 64
NHEAD         = 2
NUM_LAYERS    = 3
DIM_FF        = 128
DROPOUT       = 0.1
LAMBDA_VAR    = 2.0
MIN_PRED_STD  = 0.05

# ── Set to True + provide path to SKIP retraining and just evaluate ────────────
LOAD_WEIGHTS      = False   # ← set True if you already have .pth weights
WEIGHTS_PATH      = 'racing_line_transformer_v8.pth'  # path after upload

# ── fixed test tracks ──────────────────────────────────────
TEST_TRACKS = [
    'Aut dromo Internacional do Algarve',
    'Aut dromo Oscar y Juan G lvez',
    'Autodromo Nazionale Monza',
    'Circuit de Monaco',
    'Circuit de Nevers Magny-Cours',
    'Indianapolis Motor Speedway',
    'Jeddah Corniche Circuit',
    'Las Vegas Street Circuit',
    'N rburgring',
    'Yas Marina Circuit',
]

all_names = sorted(track_data.keys())

# Two-phase match: LCS for non-truncated, alphabetical fallback for truncated (Windows ~ names)
def _match_test_tracks(pred_names, csv_names):
    remaining = list(pred_names)
    non_trunc = [c for c in csv_names if not is_truncated(c)]
    trunc     = sorted([c for c in csv_names if is_truncated(c)])
    matched   = {}
    for ct in non_trunc:
        if not remaining: break
        best_pt = max(remaining, key=lambda pt: lcs_len(pt, ct))
        if lcs_len(best_pt, ct) >= 4:
            matched[best_pt] = ct
            remaining.remove(best_pt)
    for pt, ct in zip(sorted(remaining), trunc):
        matched[pt] = ct
    return matched

_test_match  = _match_test_tracks(TEST_TRACKS, all_names)
test_names   = [t for t in TEST_TRACKS if t in _test_match]
_matched_csvs = set(_test_match.values())
train_names  = [n for n in all_names if n not in _matched_csvs]

print(f'All loaded tracks   : {len(all_names)}')
print(f'Train tracks        : {len(train_names)}')
print(f'Test  tracks        : {len(test_names)}')

if len(test_names) == 0:
    print()
    print('⚠️  WARNING: 0 test tracks matched!')
    print('   Loaded track names (first 5):', all_names[:5])
    print('   Expected names  (first 5)   :', TEST_TRACKS[:5])
    raise ValueError('Test set is empty — check that CSV filenames match TEST_TRACKS list above.')
else:
    print(f'\n✓ Test track → CSV mapping:')
    for t in test_names: print(f'    "{t}" → "{_test_match[t]}"')

missing = [t for t in TEST_TRACKS if t not in track_data]
if missing:
    print(f'\n⚠️  Missing test tracks (not uploaded): {missing}')


## Step 3 — Feature Engineering (v8: adds curvature)

In [ ]:
def compute_features(df):
    """
    Input features (v8):
      0  local_x      – metres east  from track centroid
      1  local_y      – metres north from track centroid
      2  delta_hdg    – heading change (proxy for corner sharpness)
      3  curvature    – |κ| at each point (KEY new feature in v8)
      4  dist_inner   – metres to inner wall
      5  dist_outer   – metres to outer wall
      6  cum_dist_n   – normalised cumulative distance [0,1]
    Target:
      alpha            – lateral offset [0,1]
    """
    REF_LAT = df['t_lat'].mean()
    MLAT = 111320.0
    MLON = 111320.0 * math.cos(math.radians(REF_LAT))

    x = (df['t_lon'].values - df['t_lon'].mean()) * MLON
    y = (df['t_lat'].values - df['t_lat'].mean()) * MLAT

    dx = np.gradient(x)
    dy = np.gradient(y)
    ddx = np.gradient(dx)
    ddy = np.gradient(dy)
    denom = (dx**2 + dy**2)**1.5
    denom = np.where(denom < 1e-8, 1e-8, denom)
    curvature = np.abs(dx*ddy - dy*ddx) / denom

    heading = np.arctan2(dy, dx)
    delta_hdg = np.diff(heading, prepend=heading[0])
    delta_hdg = (delta_hdg + np.pi) % (2*np.pi) - np.pi  # wrap to [-π, π]

    cum_dist = df['cum_dist'].values if 'cum_dist' in df.columns else np.arange(len(df))
    cum_dist_n = (cum_dist - cum_dist.min()) / (cum_dist.max() - cum_dist.min() + 1e-9)

    dist_inner = df['dist_inner'].values if 'dist_inner' in df.columns else np.full(len(df), TRACK_WIDTH/2)
    dist_outer = df['dist_outer'].values if 'dist_outer' in df.columns else np.full(len(df), TRACK_WIDTH/2)

    features = np.stack([x, y, delta_hdg, curvature, dist_inner, dist_outer, cum_dist_n], axis=1)
    alpha    = df['alpha'].values

    return features.astype(np.float32), alpha.astype(np.float32)

INPUT_SIZE = 7  # v8: 7 features (was 3)

# Verify on first track
sample_name = list(track_data.keys())[0]
sample_feat, sample_alpha = compute_features(track_data[sample_name])
print(f'Feature shape: {sample_feat.shape}  (800 x {INPUT_SIZE})')
print(f'Alpha  range:  [{sample_alpha.min():.3f}, {sample_alpha.max():.3f}]  mean={sample_alpha.mean():.3f}  std={sample_alpha.std():.3f}')


## Step 4 — Mirroring

In [ ]:
def mirror_track(df):
    """Flip alpha: α_mirror = 1 - α  (left↔right).
    Flip lon/lat symmetrically around track centroid."""
    df_m = df.copy()
    df_m['alpha'] = 1.0 - df['alpha']
    # Mirror longitude around centroid (approximation sufficient for data aug)
    lon_c = df['t_lon'].mean()
    df_m['t_lon'] = 2*lon_c - df['t_lon']
    if 'rl_lon' in df.columns:
        df_m['rl_lon'] = 2*lon_c - df['rl_lon']
    # Flip inner/outer walls
    if 'dist_inner' in df.columns and 'dist_outer' in df.columns:
        df_m['dist_inner'] = df['dist_outer']
        df_m['dist_outer'] = df['dist_inner']
    return df_m

# Build augmented training set
augmented_train = {}
for name in train_names:
    augmented_train[name] = track_data[name]
    augmented_train[name + ' (mirrored)'] = mirror_track(track_data[name])

print(f'Original train tracks:   {len(train_names)}')
print(f'After mirroring:         {len(augmented_train)}')
print(f'Training points:         {len(augmented_train) * 800:,}')


## Step 5 — Scaler

In [ ]:
# Fit scaler on training data only
all_train_feats = np.vstack([compute_features(df)[0] for df in augmented_train.values()])
scaler = StandardScaler()
scaler.fit(all_train_feats)

print('Scaler fitted on training features')
print(f'Feature means:  {scaler.mean_.round(3)}')
print(f'Feature stds:   {scaler.scale_.round(3)}')

# Save scaler stats for reproducibility
scaler_stats = {
    'mean':  scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist(),
    'feature_names': ['local_x','local_y','delta_hdg','curvature','dist_inner','dist_outer','cum_dist_n']
}
with open('scaler_stats.json','w') as f:
    json.dump(scaler_stats, f, indent=2)
print('Saved scaler_stats.json')


## Step 6 — Dataset & DataLoader

In [ ]:
class TrackDataset(Dataset):
    def __init__(self, track_dict, scaler, window=WINDOW):
        self.seqs   = []
        self.labels = []
        for df in track_dict.values():
            feats, alpha = compute_features(df)
            feats = scaler.transform(feats)
            N = len(feats)
            for i in range(N):
                lo = max(0, i - window)
                hi = min(N, i + window + 1)
                seq = feats[lo:hi]
                pad_l = (i - window) < 0
                pad_r = (i + window + 1) > N
                if pad_l:
                    seq = np.vstack([np.zeros((window - i, feats.shape[1])), seq])
                if pad_r:
                    seq = np.vstack([seq, np.zeros((i + window + 1 - N, feats.shape[1]))])
                self.seqs.append(seq.astype(np.float32))
                self.labels.append(alpha[i])
        self.seqs   = np.array(self.seqs)   # (N_total, seq_len, input_size)
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):  return len(self.labels)
    def __getitem__(self, idx):
        return torch.tensor(self.seqs[idx]), torch.tensor(self.labels[idx])

train_ds = TrackDataset(augmented_train, scaler)
print(f'Train samples: {len(train_ds):,}')
print(f'Sequence shape: {train_ds.seqs.shape}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'Batches per epoch: {len(train_loader)}')


## Step 7 — Model Architecture (v8: smaller Transformer)

In [ ]:
class RacingLineTransformer(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=NUM_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        center = x[:, SEQ_LEN//2, :]
        return self.head(center).squeeze(-1)

# Reset seed before weight initialisation for full reproducibility
torch.manual_seed(SEED)
model = RacingLineTransformer().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

if LOAD_WEIGHTS:
    # ── Upload existing .pth and skip training ─────────────────────────────────
    from google.colab import files as _fw
    print(f'\nUpload your saved weights file ({WEIGHTS_PATH}):')
    w_uploaded = _fw.upload()
    w_fname = list(w_uploaded.keys())[0]
    model.load_state_dict(torch.load(w_fname, map_location=device))
    model.eval()
    print(f'✓ Weights loaded from {w_fname} — skip Steps 8 & 9, go straight to Step 10.')
else:
    print('LOAD_WEIGHTS=False — will train from scratch in Steps 8 & 9.')


## Step 8 — Training with Variance Loss (v8 key change)

In [ ]:
if not LOAD_WEIGHTS:
    # Seed before training for reproducible weight updates
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    def variance_aware_loss(y_pred, y_true, lambda_var=LAMBDA_VAR, min_std=MIN_PRED_STD):
        mae     = torch.mean(torch.abs(y_pred - y_true))
        pred_std = y_pred.std()
        var_pen  = torch.clamp(min_std - pred_std, min=0.0)
        return mae + lambda_var * var_pen, mae.item(), pred_std.item()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    history = {'loss':[], 'mae':[], 'pred_std':[], 'lr':[]}

    print(f'Training for {EPOCHS} epochs on {len(train_ds):,} samples ...')
    print(f'Loss = MAE + {LAMBDA_VAR} × max(0, {MIN_PRED_STD} - pred_std)')
    print('-'*70)

    for epoch in range(1, EPOCHS+1):
        model.train()
        ep_loss, ep_mae, ep_std = [], [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss, mae, pstd = variance_aware_loss(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss.append(loss.item()); ep_mae.append(mae); ep_std.append(pstd)
        scheduler.step()
        m_loss = np.mean(ep_loss); m_mae = np.mean(ep_mae); m_std = np.mean(ep_std)
        lr_now = scheduler.get_last_lr()[0]
        history['loss'].append(m_loss); history['mae'].append(m_mae)
        history['pred_std'].append(m_std); history['lr'].append(lr_now)
        if epoch % 10 == 0 or epoch == 1:
            flag = '⚠️ COLLAPSED' if m_std < 0.01 else ('✓' if m_std > 0.05 else '△')
            print(f'Epoch {epoch:3d}/{EPOCHS} | loss={m_loss:.4f}  mae={m_mae:.4f}  pred_std={m_std:.4f} {flag} | lr={lr_now:.2e}')

    print('\nTraining complete.')
else:
    print('Skipping training (LOAD_WEIGHTS=True).')


## Step 9 — Training Curves

In [ ]:
if not LOAD_WEIGHTS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['loss'], label='Total loss')
    axes[0].plot(history['mae'],  label='MAE component')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['pred_std'], color='orange')
    axes[1].axhline(MIN_PRED_STD, color='red', linestyle='--', label=f'min_std={MIN_PRED_STD}')
    axes[1].set_title('Predicted std (collapse monitor)'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True)
    axes[2].plot(history['lr'], color='green')
    axes[2].set_title('Learning rate'); axes[2].set_xlabel('Epoch'); axes[2].set_yscale('log'); axes[2].grid(True)
    plt.suptitle('Phase 2 v8 — Training Curves', fontsize=13)
    plt.tight_layout()
    plt.savefig('training_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved training_curve.png')
else:
    print('Skipped (weights were loaded).')


## Step 10 — Evaluation on Test Tracks

In [ ]:
model.eval()
results = []
predictions_out = {}

print('=== Test Track Evaluation ===')
print(f'{'Track':<45} | MAE_α   | RMSE_α  | err_cm  | pred_std | true_std')
print('-'*90)

for track_name in test_names:
    csv_name = _test_match.get(track_name, track_name)  # resolve truncated/mismatched names
    df = track_data[csv_name]
    feats, alpha_true = compute_features(df)
    feats_scaled = scaler.transform(feats)

    seqs = []
    N = len(feats_scaled)
    for i in range(N):
        lo, hi = max(0, i-WINDOW), min(N, i+WINDOW+1)
        seq = feats_scaled[lo:hi]
        if (i-WINDOW) < 0:
            seq = np.vstack([np.zeros((WINDOW-i, INPUT_SIZE)), seq])
        if (i+WINDOW+1) > N:
            seq = np.vstack([seq, np.zeros((i+WINDOW+1-N, INPUT_SIZE))])
        seqs.append(seq)
    seqs_t = torch.tensor(np.array(seqs, dtype=np.float32)).to(device)

    with torch.no_grad():
        alpha_pred = model(seqs_t).cpu().numpy()

    mae  = float(np.mean(np.abs(alpha_true - alpha_pred)))
    rmse = float(np.sqrt(np.mean((alpha_true - alpha_pred)**2)))
    err_cm = mae * TRACK_WIDTH * 100

    results.append({'track': track_name, 'mae': round(mae,6), 'rmse': round(rmse,6), 'err_cm': round(err_cm,2)})
    predictions_out[track_name] = {'y_true': alpha_true.tolist(), 'y_pred': alpha_pred.tolist()}

    print(f'{track_name:<45} | {mae:.4f}  | {rmse:.4f}  | {err_cm:6.1f} cm | {alpha_pred.std():.4f}    | {alpha_true.std():.4f}')

maes  = [r['mae']     for r in results]
rmses = [r['rmse']    for r in results]
cms   = [r['err_cm']  for r in results]
print('-'*90)
print(f"{'MEAN':<45} | {np.mean(maes):.4f}  | {np.mean(rmses):.4f}  | {np.mean(cms):6.1f} cm")
mean_cm = np.mean(cms)
if mean_cm < 70:
    print(f'\n🏆 EXCELLENT run! {mean_cm:.1f} cm — save this .pth file immediately!')
elif mean_cm < 100:
    print(f'\n✓ Decent run ({mean_cm:.1f} cm) — try next seed for better result')
else:
    print(f'\n✗ Bad run ({mean_cm:.1f} cm) — change SEED in Step 2 and retrain')
print(f'{'STD':<45} | {np.std(maes):.4f}  | {np.std(rmses):.4f}  | {np.std(cms):6.1f} cm')


## Step 11 — Per-Track Alpha Plots

In [ ]:
for track_name in test_names:
    y_true = np.array(predictions_out[track_name]['y_true'])
    y_pred = np.array(predictions_out[track_name]['y_pred'])
    idx    = np.arange(len(y_true))

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(idx, y_true, 'b-',  alpha=0.7, label='Ground truth α')
    axes[0].plot(idx, y_pred, 'r--', alpha=0.8, label='Predicted α')
    axes[0].axhline(0.5, color='gray', linestyle=':', linewidth=0.8, label='Centreline')
    axes[0].set_ylabel('Alpha [0=inner, 1=outer]')
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].legend(loc='upper right'); axes[0].grid(True, alpha=0.3)
    axes[0].set_title(f'{track_name} — Alpha Comparison (v8)')

    error = y_pred - y_true
    axes[1].fill_between(idx, error, alpha=0.4, color='red')
    axes[1].axhline(0, color='black', linewidth=0.8)
    mae_t = np.mean(np.abs(error))
    axes[1].set_ylabel('Prediction error (pred − true)')
    axes[1].set_xlabel('Track point index')
    axes[1].set_title(f'MAE = {mae_t:.4f}  ({mae_t*TRACK_WIDTH*100:.1f} cm)  |  pred_std = {y_pred.std():.4f}')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    safe = track_name.replace(' ','_').replace('/','_')
    plt.savefig(f'pred_{safe}.png', dpi=130, bbox_inches='tight')
    plt.show()
    plt.close()


## Step 12 — Save All Outputs

In [ ]:
# ── Save predictions.json ─────────────────────────────────
with open('predictions.json','w') as f:
    json.dump(predictions_out, f)
print('Saved predictions.json')

# ── Save phase2_results.json ──────────────────────────────
summary = {
    'model': 'RacingLineTransformer_v8',
    'random_seed': SEED,
    'input': 'local_xy + delta_hdg + curvature + dist_inner + dist_outer + cum_dist_n',
    'output': 'alpha',
    'window': WINDOW,
    'seq_len': SEQ_LEN,
    'n_train_tracks': len(augmented_train),
    'n_test_tracks': len(test_names),
    'train_tracks': list(augmented_train.keys()),
    'test_tracks': test_names,
    'per_track': results,
    'mean_mae': round(float(np.mean(maes)), 6),
    'mean_rmse': round(float(np.mean(rmses)), 6),
    'std_mae': round(float(np.std(maes)), 6),
    'std_rmse': round(float(np.std(rmses)), 6),
    'mean_err_cm': round(float(np.mean(cms)), 2),
}
with open('phase2_results.json','w') as f:
    json.dump(summary, f, indent=2)
print('Saved phase2_results.json')

# ── Save model weights ────────────────────────────────────
torch.save(model.state_dict(), 'racing_line_transformer_v8.pth')
print('Saved racing_line_transformer_v8.pth')

print(f'\n=== FINAL RESULT ===')
print(f'Mean MAE   : {summary["mean_mae"]:.6f}')
print(f'Mean err   : {summary["mean_err_cm"]:.1f} cm  (on {TRACK_WIDTH}m track)')
print(f'Std MAE    : {summary["std_mae"]:.6f}')


## Step 13 — Download All Files

In [ ]:
from google.colab import files as _dl
import glob

to_download = [
    'predictions.json',
    'phase2_results.json',
    'racing_line_transformer_v8.pth',
    'scaler_stats.json',
    'training_curve.png',
] + glob.glob('pred_*.png')

for f in to_download:
    if os.path.exists(f):
        print(f'Downloading {f} ...')
        _dl.download(f)

print('\nAll done. Upload predictions.json + phase2_results.json + test CSVs to Validation notebook.')
